In [ ]:
import re
import pandas as pd

def parse_log_file(log_file_path, output_csv_path):
    # Initialize lists to store the data
    epochs = []
    selected_clients = []
    accuracies = []
    losses = []
    run_times = []

    # Regular expressions to extract the required information
    epoch_pattern = re.compile(r'\| current_epoch (\d+) \|')
    client_select_pattern = re.compile(r'SchedulerThread select\( \d+ clients\):\s*([\d\s\|]+)')
    epoch_result_pattern = re.compile(r'Epoch\(t\): (\d+) accuracy: ([\d.]+) loss ([\d.]+) run_time: ([\d.]+)')

    # Read the log file
    with open(log_file_path, 'r') as file:
        log_content = file.read()

    # Split the log content into lines
    log_lines = log_content.splitlines()

    # Iterate through the log lines to extract the required information
    for i, line in enumerate(log_lines):
        epoch_match = epoch_pattern.search(line)
        if epoch_match:
            current_epoch = int(epoch_match.group(1))
            epochs.append(current_epoch)

            # Look ahead for the client selection line
            for j in range(i+1, min(i+10, len(log_lines))):
                client_select_match = client_select_pattern.search(log_lines[j])
                if client_select_match:
                    clients = client_select_match.group(1).strip().split(' | ')
                    selected_clients.append(clients)
                    break

            # Look ahead for the epoch result line
            for j in range(i+1, min(i+20, len(log_lines))):
                epoch_result_match = epoch_result_pattern.search(log_lines[j])
                if epoch_result_match:
                    accuracy = float(epoch_result_match.group(2))
                    loss = float(epoch_result_match.group(3))
                    run_time = float(epoch_result_match.group(4))
                    accuracies.append(accuracy)
                    losses.append(loss)
                    run_times.append(run_time)
                    break

    # Create a DataFrame from the extracted data
    data = {
        'epoch': epochs,
        'selected_clients': selected_clients,
        'accuracy': accuracies,
        'loss': losses,
        'run_time': run_times
    }
    df = pd.DataFrame(data)

    # Save the DataFrame to a CSV file
    df.to_csv(output_csv_path, index=False)

# Example usage
log_file_path = 'output (4).log'  # Replace with the actual log file path
output_csv_path = 'epoch_results.csv'  # Replace with the desired output CSV file path
parse_log_file(log_file_path, output_csv_path)
